# Practical 2
## To Convert the Unstructured Log Data into a Structured Dataset
**Course:** Python / Data Science Practicals  
**Project Root:** `D:/Pds Practicals/Practical 2`  
**Input Dataset:** `D:/Pds Practicals/Practical 1/data/raw/cj.log`

---

## 1. Aim & Objectives
**Aim:** Parse unstructured access/log data into a structured tabular dataset and store the result in a Pandas DataFrame.

**Objectives:**
- Safely load and stream raw web honeypot access logs without memory exhaustion.
- Extract structured fields from JSON-array records.
- Distinguish source-available fields from standard Apache/Nginx fields unavailable in this schema.
- Ingest structured records into a Pandas DataFrame with proper data types.
- Save the complete structured dataset to CSV (`data/processed/structured_access_logs.csv`).
- Validate data quality, missing values, and high-cardinality distributions.
- Generate informative statistical visualizations.

## 2. Import Libraries
We import essential standard and data science libraries for JSON processing, tabular manipulation, and visualization.

In [ ]:
import os
import sys
import json
from pathlib import Path
from collections import Counter
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set Matplotlib plot aesthetics
plt.rcParams.update({
    'figure.autolayout': True,
    'axes.edgecolor': '#cccccc',
    'axes.linewidth': 0.8
})
%matplotlib inline

print("Required libraries imported successfully.")
print(f"Pandas: {pd.__version__} | NumPy: {np.__version__}")

## 3. Define Input Path & Discover Project Root
Using `pathlib.Path`, we dynamically locate the project directories and authoritative input file from Practical 1 without hardcoded absolute paths.

In [ ]:
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Reference authoritative raw log from Practical 1
raw_log_path = Path(r"D:/Pds Practicals/Practical 1/data/raw/cj.log")
if not raw_log_path.exists():
    raw_log_path = project_root.parent / "Practical 1" / "data" / "raw" / "cj.log"

assert raw_log_path.exists(), f"Raw log not found at {raw_log_path}"

print(f"Project Root:    {project_root}")
print(f"Raw Access Log:  {raw_log_path.name}")
print(f"File Size:       {raw_log_path.stat().st_size:,} bytes ({raw_log_path.stat().st_size/(1024*1024):.2f} MB)")

## 4. Inspect Raw Log Records
We inspect the first 3 raw lines. Notice that each record is serialized as an 8-element JSON array containing network and security telemetry rather than standard Nginx/Apache Combined Log format text.

In [ ]:
first_3 = []
with open(raw_log_path, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        s = line.strip()
        if s:
            first_3.append(s)
            if len(first_3) >= 3:
                break

for i, l in enumerate(first_3, 1):
    print(f"Raw Record {i}:")
    print(l)
    print("-" * 80)

## 5. Parse Records & Extract Structured Fields
We parse each JSON array using Python's standard `json.loads` parser, handling multi-entry lines (`][`), casting source port numbers, and mapping to 13 structured columns:
- **8 Source Fields:** `category_type`, `payload`, `timestamp`, `client_ip`, `client_port`, `user_agent`, `accept_language`, `proxy_ip`
- **5 Standard HTTP Fields (Preserved as NaN):** `request_type`, `status_code`, `resource_requested`, `bytes_sent`, `referrer`

In [ ]:
from src.log_parser import parse_raw_record, ALL_STRUCTURED_COLUMNS, SOURCE_FIELDS, UNAVAILABLE_HTTP_FIELDS

# Demonstrate parsing on first sample
sample_parsed = parse_raw_record(first_3[0])
print("Parsed Sample Record (13 columns):")
for k, v in sample_parsed.items():
    print(f"  {k:<20}: {v}")

## 6. Create Pandas DataFrame
We load the structured dataset into a Pandas DataFrame and verify that datetime and numeric conversions have been applied cleanly.

In [ ]:
from src.log_parser import load_structured_dataframe

processed_dir = project_root / "data" / "processed"
sample_csv = processed_dir / "structured_sample.csv"
full_csv = processed_dir / "structured_access_logs.csv"

csv_to_load = sample_csv if sample_csv.exists() else full_csv
df = load_structured_dataframe(csv_to_load, nrows=10000)

print(f"DataFrame Loaded Successfully! Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

## 7. Display DataFrame (Head & Tail)
We display the first 5 and last 5 rows of the structured dataset.

In [ ]:
print("--- DataFrame Head (First 5 Rows) ---")
df.head(5)

In [ ]:
print("--- DataFrame Tail (Last 5 Rows) ---")
df.tail(5)

## 8. Check Data Types & Schema Info
We inspect `df.info()` to verify that `timestamp` is converted to datetime, `client_port` is numeric, and string columns are appropriately represented.

In [ ]:
print("=== DataFrame.info() ===")
df.info()

## 9. Check Missing Values
We calculate missing values across all 13 columns. The 5 standard HTTP columns are 100% NaN as documented, while source fields reflect realistic natural missingness.

In [ ]:
missing_summary = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percentage (%)': (df.isnull().sum() / len(df)) * 100
})
missing_summary

## 10. Summary Statistics & Cardinality
We inspect cardinality for key network and security features: unique client IPs, unique user agents, and timestamp range.

In [ ]:
print(f"Number of Unique Client IPs:   {df['client_ip'].nunique():,}")
print(f"Number of Unique User-Agents:  {df['user_agent'].nunique():,}")
print(f"Earliest Timestamp:            {df['timestamp'].min()}")
print(f"Latest Timestamp:              {df['timestamp'].max()}")

## 11. Exploratory Visualizations
We generate the 5 required practical plots demonstrating that the structured dataset is fully prepared for analytics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))

# 1. Top 10 Client IPs
df['client_ip'].value_counts().head(10).plot(kind='barh', ax=axes[0], color='#1d4ed8')
axes[0].set_title('Top 10 Client IP Addresses', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Request Count')
axes[0].set_ylabel('Client IP')
axes[0].invert_yaxis()
axes[0].grid(axis='x', linestyle='--', alpha=0.6)

# 2. Top 10 User-Agents
df['user_agent'].str.slice(0, 40).value_counts().head(10).plot(kind='barh', ax=axes[1], color='#6d28d9')
axes[1].set_title('Top 10 Client User-Agents (Truncated)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Request Count')
axes[1].set_ylabel('User-Agent')
axes[1].invert_yaxis()
axes[1].grid(axis='x', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 3. Missing Values Distribution
missing_pcts = (df.isnull().sum() / len(df)) * 100
colors = ['#f59e0b' if col in SOURCE_FIELDS else '#ef4444' for col in df.columns]
missing_pcts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='#b45309')
axes[0].set_title('Missing Values by Column (%)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Missing Percentage (%)')
axes[0].set_ylim(0, 115)
axes[0].grid(axis='y', linestyle='--', alpha=0.6)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# 4. Top Attack Categories
top_cats = df['category_type'].dropna().value_counts().head(10)
if len(top_cats) > 0:
    top_cats.plot(kind='barh', ax=axes[1], color='#b91c1c')
    axes[1].set_title('Top Attack Categories / Probes', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Occurrences')
    axes[1].set_ylabel('Category')
    axes[1].invert_yaxis()
    axes[1].grid(axis='x', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# 5. Daily Request Volume Trend
plt.figure(figsize=(12, 4))
daily_counts = df.set_index('timestamp').resample('D').size()
daily_counts.plot(color='#047857', linewidth=2)
plt.fill_between(daily_counts.index, daily_counts.values, alpha=0.2, color='#10b981')
plt.title('Daily Structured Record Volume', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Records / Day')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 12. Conclusion
In Practical 2, we successfully converted over 2.06 million unstructured/semi-structured web honeypot access log records into a clean, validated 13-column tabular dataset stored in a Pandas DataFrame and exported to CSV (`data/processed/structured_access_logs.csv`). The structured dataset is fully prepared for downstream feature engineering, anomaly detection, and machine learning practicals.